# 06_Unsupervised_Model_Development_LDA

In [15]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import LatentDirichletAllocation
import polars as pl
import joblib


In [3]:
# Load the parquet file
INPUT_PATH = '../data/processed/clustered_narratives.parquet'
df_narratives = pd.read_parquet(
    INPUT_PATH,
    columns=[
        'Complaint ID',
        'processed_narrative'
    ]
)

In [4]:
df_narratives.head()

,Complaint ID,processed_narrative
0,3442136,claimed delivered package address never receiv...
1,3601853,got brink money pre paid card mail assuming un...
2,3300820,called creditor nelson cruz associate claimed ...
3,3739698,around opened credit card account online capit...
4,3619130,rushmore loan management permit applying loan ...


In [5]:
# Run Count Vectorizer on 50k sample to save memory
df_lda_sample = df_narratives.sample(
    n=50_000,
    random_state=42
)

count_vectorizer = CountVectorizer(
    min_df=20,
    max_df=0.90,
    max_features=10_000
)

X_count = count_vectorizer.fit_transform(
    df_lda_sample['processed_narrative']
)

print(X_count.shape)

(50000, 6455)


In [6]:
# Run LDA on sample
lda = LatentDirichletAllocation(
    n_components=10,
    learning_method='online',
    random_state=42,
    n_jobs=-1,
    batch_size=2048
)

lda.fit(X_count)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",10
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'online'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",2048
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [7]:
# Print top terms in the LDA topics
feature_names = np.array(count_vectorizer.get_feature_names_out())

for topic_idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[::-1][:15]
    top_terms = feature_names[top_indices]

    print(f'\nTopic {topic_idx}')
    print(', '.join(top_terms))


Topic 0
account, complaint, financial, credit, bank, request, information, consumer, provide, issue, matter, despite, action, regarding, without

Topic 1
charge, dispute, transaction, claim, card, chase, merchant, refund, received, purchase, made, amount, fraud, time, service

Topic 2
payment, fee, late, interest, balance, account, month, due, pay, statement, paid, made, amount, charge, time

Topic 3
account, bank, money, told, would, called, call, back, said, day, time, get, could, phone, number

Topic 4
account, credit, one, capital, report, information, card, closed, number, address, name, fraud, opened, identity, never

Topic 5
card, credit, would, time, citi, told, called, account, customer, received, purchase, service, point, citibank, said

Topic 6
well, fargo, loan, mortgage, modification, foreclosure, document, home, sale, property, letter, attorney, court, servicing, complaint

Topic 7
check, account, fund, bank, fee, deposit, transaction, overdraft, balance, checking, trans

In [8]:
# Add dominant topic to the sample dataframe
topic_probs = lda.transform(X_count)

df_lda_sample['dominant_topic'] = topic_probs.argmax(axis=1)
df_lda_sample['topic_probability'] = topic_probs.max(axis=1)

df_lda_sample['dominant_topic'].value_counts().sort_index()

dominant_topic
0     3565
1     4962
2     5283
3    12950
4     3820
5     5602
6     2527
7     2889
8     7317
9     1085
Name: count, dtype: int64

In [9]:
df_lda_sample.head()

,Complaint ID,processed_narrative,dominant_topic,topic_probability
411174,7840221,approximately deposited well fargo atm atm tak...,3,0.758217
172423,17318283,year received email capital one name credit ca...,4,0.520913
217642,9042644,year prepaid hotel app stay year year confirme...,1,0.789616
191177,8311223,fraudulent account set name attempted cancel a...,3,0.389431
53751,8210071,see attached document want bureau commence inv...,4,0.825783


In [2]:
# for topic in sorted(df_lda_sample['dominant_topic'].unique()):
#     print(f'\n===== Topic {topic} =====')
#
#     samples = (
#         df_lda_sample[df_lda_sample['dominant_topic'] == topic]
#         .sample(n=3, random_state=42)
#     )
#
#     for text in samples['processed_narrative']:
#         print('\n', text[:500])

In [10]:
# Load in cleaned narratives for the Complaint ID's in the LDA 50k sample
sample_ids = df_lda_sample['Complaint ID'].to_list()

df_readable = (
    pl.scan_parquet(INPUT_PATH)
    .filter(pl.col('Complaint ID').is_in(sample_ids))
    .select([
        'Complaint ID',
        'cleaned_consumer_narrative'
    ])
    .collect()
    .to_pandas()
)

df_lda_sample = df_lda_sample.merge(
    df_readable,
    on='Complaint ID',
    how='left'
)

In [11]:
# Print and inspect example narratives in each topic
for topic in sorted(df_lda_sample['dominant_topic'].unique()):
    print(f'\n===== Topic {topic} =====')

    samples = (
        df_lda_sample[df_lda_sample['dominant_topic'] == topic]
        .sample(n=3, random_state=42)
    )

    for text in samples['cleaned_consumer_narrative']:
        print('\n', text[:500])


===== Topic 0 =====

 Dear Fraud Investigation Department, I am writing to formally demand that Elan Financial Services provide immediate and complete documentation and resolution concerning fraudulent charges on my accounts ending in REDACTED and REDACTED . Despite prior confirmations of fraud, Elan reversed credits for the following transactions : - {$620.00} at REDACTED # REDACTED on REDACTED / REDACTED /year> - {$2000.00} at REDACTED on REDACTED / REDACTED /year> These were previously deemed fraudulent, and my ac

 Formal Complaint Regarding Unauthorized Charges Request for Assistance Dear Consumer Financial Protection Bureau ( CFPB ), I am writing to formally file a complaint regarding multiple unauthorized charges made to my bank account between REDACTED_DATE and REDACTED / REDACTED /year>, totaling over {$40000.00}. I reported the issue promptly to Truist Bank and provided all requested documentation to support their investigation. However, as of today, I have not received a re

In [12]:
# Add Topic Probabilities as a Feature
topic_features = pd.DataFrame(
    topic_probs,
    columns=[
        f'topic_{i}_prob'
        for i in range(topic_probs.shape[1])
    ]
)

df_lda_sample = pd.concat(
    [df_lda_sample.reset_index(drop=True),
     topic_features.reset_index(drop=True)],
    axis=1
)

df_lda_sample.head()

,Complaint ID,processed_narrative,dominant_topic,topic_probability,cleaned_consumer_narrative,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,topic_8_prob,topic_9_prob
0,7840221,approximately deposited well fargo atm atm tak...,3,0.758217,"On REDACTED / REDACTED /2023, at approximately...",0.039711,0.128287,0.000827,0.758217,0.000827,0.000827,0.000827,0.068826,0.000827,0.000827
1,17318283,year received email capital one name credit ca...,4,0.520913,"On REDACTED / REDACTED /year>, I received an e...",0.000827,0.000827,0.000827,0.000827,0.520913,0.472474,0.000827,0.000827,0.000827,0.000827
2,9042644,year prepaid hotel app stay year year confirme...,1,0.789616,"On REDACTED / REDACTED /year>, I prepaid for a...",0.001389,0.789616,0.001389,0.080926,0.001389,0.119733,0.001389,0.001389,0.001389,0.001389
3,8311223,fraudulent account set name attempted cancel a...,3,0.389431,A fraudulent account was set up in my name and...,0.003335,0.135670,0.003334,0.389431,0.302376,0.003334,0.152517,0.003334,0.003334,0.003334
4,8210071,see attached document want bureau commence inv...,4,0.825783,See the attached documents. I want the bureau ...,0.007693,0.007693,0.007695,0.007693,0.825783,0.007693,0.112672,0.007693,0.007693,0.007693


In [14]:
# One-hot encode Dominant Topic
topic_dummies = pd.get_dummies(
    df_lda_sample['dominant_topic'],
    prefix='dominant_topic'
)

df_lda_sample = pd.concat(
    [df_lda_sample, topic_dummies],
    axis=1
)

df_lda_sample.head()

,Complaint ID,processed_narrative,dominant_topic,topic_probability,cleaned_consumer_narrative,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,...,dominant_topic_0,dominant_topic_1,dominant_topic_2,dominant_topic_3,dominant_topic_4,dominant_topic_5,dominant_topic_6,dominant_topic_7,dominant_topic_8,dominant_topic_9
0,7840221,approximately deposited well fargo atm atm tak...,3,0.758217,"On REDACTED / REDACTED /2023, at approximately...",0.039711,0.128287,0.000827,0.758217,0.000827,...,False,False,False,True,False,False,False,False,False,False
1,17318283,year received email capital one name credit ca...,4,0.520913,"On REDACTED / REDACTED /year>, I received an e...",0.000827,0.000827,0.000827,0.000827,0.520913,...,False,False,False,False,True,False,False,False,False,False
2,9042644,year prepaid hotel app stay year year confirme...,1,0.789616,"On REDACTED / REDACTED /year>, I prepaid for a...",0.001389,0.789616,0.001389,0.080926,0.001389,...,False,True,False,False,False,False,False,False,False,False
3,8311223,fraudulent account set name attempted cancel a...,3,0.389431,A fraudulent account was set up in my name and...,0.003335,0.135670,0.003334,0.389431,0.302376,...,False,False,False,True,False,False,False,False,False,False
4,8210071,see attached document want bureau commence inv...,4,0.825783,See the attached documents. I want the bureau ...,0.007693,0.007693,0.007695,0.007693,0.825783,...,False,False,False,False,True,False,False,False,False,False


In [16]:
# Write fitted Count Vectorizer and LDA to file
joblib.dump(
    count_vectorizer,
    'count_vectorizer.pkl'
)

joblib.dump(
    lda,
    'lda_model.pkl'
)

['lda_model.pkl']